In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("enron_spam_data.csv")

In [4]:
df.head(10)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,1999-12-14
6,6,meter 1517 - jan 1999,"george ,\ni need the following done :\njan 13\...",ham,1999-12-14
7,7,duns number changes,fyi\n- - - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
8,8,king ranch,there are two fields of gas that i am having d...,ham,1999-12-14
9,9,re : entex transistion,thanks so much for the memo . i would like to ...,ham,1999-12-14


In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (33716, 5)
Columns: ['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date']


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [7]:
df.isnull().sum()

Message ID      0
Subject       289
Message       371
Spam/Ham        0
Date            0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(3222)

In [20]:
df = pd.read_csv("enron_spam_data.csv")

df.drop(columns=["Message ID", "Date", "Subject"], inplace=True)

In [21]:
df["Spam/Ham"] = df["Spam/Ham"].map({"ham": 0, "spam": 1})
df.dropna(subset=["Message"], inplace=True)

df.head()

,Message,Spam/Ham
1,"gary , production from the high island larger ...",0
2,- calpine daily gas nomination 1 . doc,0
3,fyi - see note below - already done .\nstella\...,0
4,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0
5,"jackie ,\nsince the inlet to 3 river plant is ...",0


## removed features:
removed un-useful features like date and message ID as seemed irrelevent for me

In [22]:
from sklearn.model_selection import train_test_split

X = df["Message"]
y = df["Spam/Ham"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (26676,)
Testing data: (6669,)


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

Training matrix: (26676, 20000)
Testing matrix: (6669, 20000)


In [24]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred = nb_model.predict(X_test_tfidf)

In [25]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.9827560353876144

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.98      3299
        Spam       0.98      0.99      0.98      3370

    accuracy                           0.98      6669
   macro avg       0.98      0.98      0.98      6669
weighted avg       0.98      0.98      0.98      6669


Confusion Matrix:
[[3230   69]
 [  46 3324]]


In [26]:
test_messages = [
    "Congratulations! You have won a free prize. Click here now!",
    "Hey, are we meeting at the library tomorrow?",
    "URGENT! Claim your cash reward now!",
    "Please send me the project report by evening.",
    "You have been selected for a lottery. Send your bank details."
]

sample_tfidf = tfidf.transform(test_messages)
predictions = nb_model.predict(sample_tfidf)
probabilities = nb_model.predict_proba(sample_tfidf)

for message, prediction, probability in zip(
    test_messages, predictions, probabilities
):
    label = "Spam" if prediction == 1 else "Ham"
    spam_probability = probability[1]

    print(f"Message: {message}")
    print(f"Prediction: {label}")
    print(f"Spam probability: {spam_probability:.2%}")
    print("-" * 60)

Message: Congratulations! You have won a free prize. Click here now!
Prediction: Spam
Spam probability: 94.14%
------------------------------------------------------------
Message: Hey, are we meeting at the library tomorrow?
Prediction: Ham
Spam probability: 17.53%
------------------------------------------------------------
Message: URGENT! Claim your cash reward now!
Prediction: Spam
Spam probability: 88.08%
------------------------------------------------------------
Message: Please send me the project report by evening.
Prediction: Ham
Spam probability: 20.93%
------------------------------------------------------------
Message: You have been selected for a lottery. Send your bank details.
Prediction: Spam
Spam probability: 95.23%
------------------------------------------------------------


In [28]:
import joblib

joblib.dump(nb_model, "spam_model.pkl")
joblib.dump(tfidf, "spam_tfidf_vectorizer.pkl")

print("Both files saved successfully!")

Both files saved successfully!
